# Scikit-learn 基础用法

本 notebook 简单介绍 Scikit-learn 的两个最基础模型：
1. **线性回归（LinearRegression）** — 用于解决回归问题
2. **逻辑回归（LogisticRegression）** — 用于解决分类问题

同时使用 `display` 来美化输出结果。

## 1. 导入所需库

In [1]:
# 基础数据处理
import numpy as np
import pandas as pd

# display 函数用于更友好地展示输出
from IPython.display import display

# 数据集工具
from sklearn.datasets import make_regression, make_classification
from sklearn.model_selection import train_test_split

# 模型
from sklearn.linear_model import LinearRegression, LogisticRegression

# 评估指标
from sklearn.metrics import mean_squared_error, accuracy_score

## 2. 线性回归示例

线性回归用于预测**连续数值**（如房价、气温）。
基本流程：生成数据 → 划分数据集 → 训练模型 → 预测 → 评估

In [2]:
# 2.1 生成回归数据集
X, y = make_regression(n_samples=200,   # 200个样本
                       n_features=1,    # 1个特征
                       noise=10,        # 添加噪声
                       random_state=0)

display(pd.DataFrame({'X': X[:, 0], 'y': y}).head())

,X,y
0,0.302472,15.090460
1,-1.270485,-137.958395
2,1.532779,140.515903
3,-1.602058,-142.235045
4,1.139401,106.781745


In [3]:
# 2.2 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0)

print('数据集划分结果：')
display(pd.DataFrame({
    '数据集': ['训练集', '测试集'],
    '样本数': [len(X_train), len(X_test)]
}))

数据集划分结果：


,数据集,样本数
0,训练集,150
1,测试集,50


In [4]:
# 2.3 创建并训练线性回归模型
model = LinearRegression()
model.fit(X_train, y_train)

# 输出模型的系数和截距
display(pd.DataFrame({
    '参数': ['斜率 (coef_)', '截距 (intercept_)'],
    '数值': [model.coef_[0], model.intercept_]
}))

,参数,数值
0,斜率 (coef_),96.608161
1,截距 (intercept_),-1.644975


In [5]:
# 2.4 预测并评估模型
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)

print('预测结果示例：')
display(pd.DataFrame({'真实值': y_test[:5], '预测值': y_pred[:5]}))

print(f'\n均方误差 (MSE): {mse:.2f}')

预测结果示例：


,真实值,预测值
0,-58.437984,-45.653132
1,50.477076,59.704244
2,84.580597,85.382215
3,-5.660037,-11.616758
4,84.664613,90.141320



均方误差 (MSE): 81.44


## 2.5 LinearRegression 的常用参数

上面的例子使用了默认参数，实际上 `LinearRegression` 还支持以下参数：

| 参数 | 默认值 | 说明 |
|------|--------|------|
| `fit_intercept` | `True` | 是否计算截距。若为 `False`，模型强制过原点 |
| `copy_X` | `True` | 是否复制 X（否则可能被覆盖） |
| `n_jobs` | `None` | 并行计算的线程数，`-1` 表示用全部 CPU |
| `positive` | `False` | 若为 `True`，强制所有系数为非负数 |

```python
# 示例：使用非默认参数
model = LinearRegression(fit_intercept=False, positive=True)
```

> 注意：`LinearRegression` 是普通最小二乘回归（OLS），**没有**正则化参数。
> 如果需要 L1/L2 正则化，可以改用 `Ridge`（L2）、`Lasso`（L1）或 `ElasticNet`（L1+L2）。


## 2.6 多项式拟合

当数据不是线性关系（例如抛物线、曲线趋势）时，可以用**多项式拟合**。

在 Scikit-learn 中的做法分两步：
1. 用 `PolynomialFeatures` 把原始特征扩展成多项式的组合（如 $x, x^2, x^3, \dots$）
2. 再套用 `LinearRegression` 进行线性拟合 —— 这时的"线性"是相对于**新特征**而言的

> 核心思想：**多项式拟合本质上是线性回归的变形**，只是先把特征升维。


In [6]:
# 2.6.1 生成非线性数据（抛物线趋势）
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

X, y = make_regression(n_samples=100,
                       n_features=1,
                       noise=10,
                       random_state=0)
# 手动构造二次关系，制造"非线性"效果
y = y ** 2

# 2.6.2 方式一：用 PolynomialFeatures + LinearRegression
poly = PolynomialFeatures(degree=2)     # 生成 x, x² 特征
X_poly = poly.fit_transform(X)          # 将原始特征升维
model = LinearRegression()
print('X_poly shape:', X_poly.shape)
print('X_poly first 5 data:', X_poly[:5,:])
model.fit(X_poly, y)
print('多项式特征名：', poly.get_feature_names_out())
print('系数：', model.coef_)

# 2.6.3 方式二（推荐）：用 Pipeline 一步完成
model_pipe = make_pipeline(PolynomialFeatures(degree=2), LinearRegression())
model_pipe.fit(X, y)
print('Pipeline 方式拟合完成！')


X_poly shape: (100, 3)
X_poly first 5 data: [[ 1.         -0.35955316  0.12927848]
 [ 1.          0.97663904  0.95382381]
 [ 1.          0.40234164  0.1618788 ]
 [ 1.         -0.81314628  0.66120688]
 [ 1.         -0.88778575  0.78816353]]
多项式特征名： ['1' 'x0' 'x0^2']
系数： [   0.         -163.25751641 1853.16247537]
Pipeline 方式拟合完成！


## 2.6.2 为什么 X_poly 有 3 列而不是 2 列？

刚才看到 `X_poly` 的 shape 是 `(100, 3)`，多出来的那一列是**常数项（偏置项）**。

`PolynomialFeatures` 默认 `include_bias=True`，对单个特征 $x$ 且 `degree=2` 时，会生成：

$$\text{X\_poly} = [\, \underbrace{1}_{\text{常数项}}, \; x, \; x^2 \,]$$

- 第 0 列：全为 1 的常数项（偏置）
- 第 1 列：$x$
- 第 2 列：$x^2$

**如果不想要常数项**，可以设置 `include_bias=False`，此时就只有 $[x, x^2]$ 两列：

```python
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X)   # shape 变为 (100, 2)
```

> 提示：`LinearRegression` 本身会拟合截距（`intercept_`）。
> 如果让 `PolynomialFeatures` 和 `LinearRegression` 同时带偏置，特征矩阵会**秩亏**，模型会给出警告。
> 通常的做法是：让 `PolynomialFeatures(include_bias=False)` 生成特征，由 `LinearRegression` 负责拟合截距。


## 2.7 关于多项式拟合的注意点

- `degree` 表示多项式最高次数：`degree=2` 生成 $[x, x^2]$，`degree=3` 生成 $[x, x^2, x^3]$，以此类推
- **次数越高，拟合越"弯曲"，但也越容易过拟合**（训练集很好、测试集很差）
- 多项式特征会**成倍增加特征数量**，当原始特征多、次数高时计算量会很大
- 实践经验：先尝试 `degree=2` 或 `degree=3`，观察测试集误差是否下降，而不是一味提高次数

> 小结：多项式拟合 = `PolynomialFeatures`（升维） + `LinearRegression`（拟合）。
> 推荐用 `make_pipeline` 把它们组合起来，避免手动转换特征的麻烦。


## 3. 逻辑回归分类示例

逻辑回归用于**分类问题**（预测离散的类别标签，如二分类的0/1）。
基本流程与回归一致：生成数据 → 划分 → 训练 → 预测 → 评估

In [7]:
# 3.1 生成分类数据集（二分类）
X, y = make_classification(n_samples=200,   # 200个样本
                           n_features=2,    # 2个特征
                           n_informative=2, # 2个有效特征
                           n_redundant=0,   # 无冗余特征
                           random_state=0)

display(pd.DataFrame({'特征1': X[:, 0], '特征2': X[:, 1], '类别': y}).head())

,特征1,特征2,类别
0,0.617011,-1.193995,1
1,0.057417,-0.572398,0
2,-0.558915,0.755356,0
3,0.368073,0.061557,1
4,2.327470,-2.928569,1


In [8]:
# 3.2 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0)

print('数据集划分结果：')
display(pd.DataFrame({
    '数据集': ['训练集', '测试集'],
    '样本数': [len(X_train), len(X_test)]
}))

数据集划分结果：


,数据集,样本数
0,训练集,150
1,测试集,50


In [9]:
# 3.3 创建并训练逻辑回归模型
clf = LogisticRegression()
clf.fit(X_train, y_train)

print('模型已训练完成！')

模型已训练完成！


In [10]:
# 3.4 预测并评估模型
y_pred = clf.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print('预测结果示例：')
display(pd.DataFrame({'真实类别': y_test[:5], '预测类别': y_pred[:5]}))

print(f'\n准确率 (Accuracy): {acc:.2f}')

预测结果示例：


,真实类别,预测类别
0,1,1
1,1,1
2,0,0
3,1,1
4,1,1



准确率 (Accuracy): 0.94


## 3.5 LogisticRegression 的常用参数

逻辑回归比线性回归的参数更多，最常调整的有：

| 参数 | 默认值 | 说明 |
|------|--------|------|
| `penalty` | `'l2'` | 正则化类型：`'l1'`、`'l2'`、`'elasticnet'`、`'none'` |
| `C` | `1.0` | 正则化强度的倒数，**越小**正则化越强（越不容易过拟合） |
| `solver` | `'lbfgs'` | 求解算法，如 `'lbfgs'`、`'liblinear'`、`'saga'`、`'newton-cg'` |
| `max_iter` | `100` | 最大迭代次数，不收敛时可调大 |
| `class_weight` | `None` | 类别权重，类别不平衡时可用 `'balanced'` |
| `multi_class` | `'auto'` | 多分类策略：`'ovr'`（一对多）或 `'multinomial'`（多项式） |

```python
# 示例：调整正则化强度、增加迭代次数
clf = LogisticRegression(penalty='l1', C=0.1, solver='liblinear', max_iter=500)
```

> 提示：`solver` 与 `penalty` 有兼容性限制（例如 `'l1'` 需要 `'liblinear'` 或 `'saga'`）。
> 如果你刚开始学习，先掌握 `C` 和 `max_iter` 这两个参数即可。


## 4. 总结

| 对比项 | 线性回归 | 逻辑回归 |
|--------|---------|---------|
| 问题类型 | 回归（连续值） | 分类（离散值） |
| 预测输出 | 数值 | 类别标签 |
| 常用评估指标 | MSE, R² | Accuracy, Precision |
| 训练接口 | `fit(X, y)` | `fit(X, y)` |
| 预测接口 | `predict(X)` | `predict(X)` |

> 两者的用法高度一致，掌握了这个基本流程，就可以迁移到 Scikit-learn 中其他更复杂的模型！

## 5. 用 TensorFlow 实现线性回归与分类

**当然可以！** Scikit-learn 和 TensorFlow 都能做线性回归/分类，但思路不同：

| 对比项 | Scikit-learn | TensorFlow (Keras) |
|--------|-------------|--------------------|
| 定位 | 传统机器学习 | 深度学习框架 |
| 建模方式 | 直接调用模型类 | 手动搭建"层"（神经网络） |
| 核心概念 | `fit` / `predict` | 模型 + 损失函数 + 优化器 |
| 线性回归 | `LinearRegression()` | 单层 `Dense(1)` 网络 |
| 逻辑回归 | `LogisticRegression()` | `Dense(1, activation='sigmoid')` |
| 适合场景 | 中小型数据、快速原型 | 大数据、深度学习、GPU 加速 |

**核心思想**：在 TensorFlow 里，线性回归就是一个"没有隐藏层、单个神经元的神经网络"。


In [11]:
# 5.1 用 TensorFlow/Keras 实现线性回归
# 注：需要先安装 tensorflow（pip install tensorflow）
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# 重新生成回归数据
X, y = make_regression(n_samples=200, n_features=1, noise=10, random_state=0)

# 搭建模型：单层、单个神经元（等价于线性回归）
tf_model = keras.Sequential([
    layers.Dense(1, activation=None)   # 1个输出，无激活函数 -> 线性回归
])

# 编译：指定损失函数和优化器
tf_model.compile(optimizer='sgd', loss='mse')

# 训练
tf_model.fit(X, y, epochs=50, verbose=0)
print('训练完成！')

# 预测
y_pred_tf = tf_model.predict(X[:5])
print('TensorFlow 预测示例：', y_pred_tf.flatten())


I0000 00:00:1786605966.150582   27731 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1786605966.157257   27731 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1786605966.769802   27731 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1786605968.742664   27731 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:0

训练完成！
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
TensorFlow 预测示例： [  27.69138  -123.83502   146.2096   -155.77614   108.314575]


In [13]:
# 5.2 用 TensorFlow/Keras 实现逻辑回归（二分类）
# 生成二分类数据
X, y = make_classification(n_samples=200, n_features=2,
                           n_informative=2, n_redundant=0, random_state=0)

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0)

# 搭建模型：单个神经元 + sigmoid 激活函数 -> 逻辑回归
tf_clf = keras.Sequential([
    layers.Dense(1, activation='sigmoid')   # 输出 0~1 之间的概率
])

# 编译：二分类用 binary_crossentropy
tf_clf.compile(optimizer='sgd', loss='binary_crossentropy', metrics=['accuracy'])

# 训练
tf_clf.fit(X_train, y_train, epochs=50, verbose=0)
print('分类模型训练完成！')

# 预测（输出概率，>0.5 判为类别 1）
prob = tf_clf.predict(X[:5])
print('预测概率示例：', prob.flatten())
print('预测类别：', (prob > 0.5).astype(int).flatten())

y_pred = clf.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f'\n准确率 (Accuracy): {acc:.2f}')

分类模型训练完成！
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
预测概率示例： [0.41548058 0.4196149  0.51511717 0.555655   0.42576453]
预测类别： [0 0 1 1 0]

准确率 (Accuracy): 0.94


## 5.3 小结：Keras 与 Scikit-learn 的对应关系

| 任务 | Scikit-learn | Keras 搭建 | 损失函数 |
|------|-------------|-----------|---------|
| 线性回归 | `LinearRegression()` | `Dense(1)` 无激活函数 | `mse` |
| 逻辑回归 | `LogisticRegression()` | `Dense(1, activation='sigmoid')` | `binary_crossentropy` |
| 多分类 | - | `Dense(k, activation='softmax')` | `categorical_crossentropy` |

> **提示**：
> - 运行 5.1 / 5.2 前需要先安装 TensorFlow：`pip install tensorflow`
> - 数据量小、问题简单时，用 Scikit-learn 更省事、结果更稳定
> - 数据量大、任务复杂（如图像、文本、深度学习）时，才需要 TensorFlow 这类深度学习框架
> - 学习路线建议：先掌握 Scikit-learn，再学 TensorFlow，会发现很多概念是相通的
